In [ ]:
import google.generativeai as genai
from PIL import Image
from google.colab import userdata

# Configure Gemini API
# Make sure your GOOGLE_API_KEY is stored in Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize Gemini Pro Vision model, using an available vision model
gemini_pro_vision_model = genai.GenerativeModel('models/gemini-3.5-flash') # Updated model name
print("Gemini Pro Vision model initialized ✅")

Gemini Pro Vision model initialized ✅


In [ ]:
# Install Tesseract-OCR engine and pytesseract Python wrapper
!sudo apt update
!sudo apt install tesseract-ocr
!pip install pytesseract

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
98 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree

In [ ]:
import io
from PIL import Image
from google.colab import files

uploaded_images = files.upload()

image_objects = []
for file_name, file_content in uploaded_images.items():
    try:
        img = Image.open(io.BytesIO(file_content))
        image_objects.append(img)
        print(f"Loaded image: {file_name}")
    except Exception as e:
        print(f"Could not load {file_name} as an image: {e}")

print(f"\nSuccessfully loaded {len(image_objects)} images.")

Saving Screenshot 2026-07-13 184252.png to Screenshot 2026-07-13 184252 (1).png
Loaded image: Screenshot 2026-07-13 184252 (1).png

Successfully loaded 1 images.


In [ ]:
import io
import pytesseract
from PIL import Image

def ask_image_llm(question: str, images: list[Image.Image]):
    if not images:
        return "No images provided for analysis."

    # Extract text from images using OCR
    ocr_texts = []
    for img in images:
        try:
            text = pytesseract.image_to_string(img)
            if text.strip(): # Only add if text is found
                ocr_texts.append(text)
        except Exception as e:
            print(f"OCR failed for an image: {e}")

    # Combine OCR text with the original question
    full_question = question
    if ocr_texts:
        full_question = "Question: " + question + "\n\nExtracted text from images:\n" + "\n---\n".join(ocr_texts)

    # Prepare content for the multimodal model
    content_parts = [full_question] + images

    response = gemini_pro_vision_model.generate_content(content_parts)
    return response.text

print("Multimodal query function updated with OCR functionality ✅")

Multimodal query function updated with OCR functionality ✅


In [ ]:
if image_objects:
    image_query = input("What would you like to ask about the image(s)? ")
    image_answer = ask_image_llm(image_query, image_objects)
    print(image_answer)

else:
    print("No images were uploaded. Please upload images in the previous step.")

What would you like to ask about the image(s)? WHEN THE PRODUCT IS ORDERED?


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash
Please retry in 13.022170844s.

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)